<a href="https://colab.research.google.com/github/IoNiCx1/GPT-Generative-Pre-trained-Transformer/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [96]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-15 15:46:56--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.3’

input.txt.3         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-08-15 15:46:56 (20.7 MB/s) - ‘input.txt.3’ saved [1115394/1115394]



In [97]:
with open('input.txt','r',encoding='utf-8') as f:
  text = f.read()

In [98]:
print("length of dataset in characters:",len(text))

length of dataset in characters: 1115394


In [99]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [100]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [101]:
stoi = {ch:i for i,ch in  enumerate(chars)}
itos = {i:ch for i,ch in  enumerate (chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("Omkar"))
print(decode(encode("Omkar")))

[27, 51, 49, 39, 56]
Omkar


In [102]:
import torch
data = torch.tensor(encode(text),dtype = torch.long)
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [103]:
n = int(0.9* len(data))
train_data= data[:n]
val_data = data[n:]

In [104]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [105]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [106]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8


def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data)- block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i:i+block_size+1] for i in ix])
  return x,y

xb,yb = get_batch('train')
print('inputs:')

print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print("----")

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(f'when input is {context.tolist()} the target:{target}')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 9])
tensor([[24, 43, 58,  5, 57,  1, 46, 43, 39],
        [44, 53, 56,  1, 58, 46, 39, 58,  1],
        [52, 58,  1, 58, 46, 39, 58,  1, 46],
        [25, 17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target:24
when input is [24, 43] the target:43
when input is [24, 43, 58] the target:58
when input is [24, 43, 58, 5] the target:5
when input is [24, 43, 58, 5, 57] the target:57
when input is [24, 43, 58, 5, 57, 1] the target:1
when input is [24, 43, 58, 5, 57, 1, 46] the target:46
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target:43
when input is [44] the target:44
when input is [44, 53] the target:53
when input is [44, 53, 56] the target:56
when input is [44, 53, 56, 1] the target:1
when input is [44, 53, 56, 1, 58] the target:58
when input is [44,

In [107]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [108]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
# 65 * 65 size of Matrix
  def forward(self,idx,targets=None):
    logits = self.token_embedding_table(idx) # (B, T, C)

    if targets is None:
        loss = None
    else:
        B, T, C = logits.shape
        # Use reshape() instead of view() for logits
        logits = logits.reshape(B*T,C)
        # Correctly slice targets to match the block_size before reshaping, using reshape()
        targets = targets[:, 1:].reshape(B*T)
        loss = F.cross_entropy(logits,targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            # The model is called here, but only 'idx' is passed, so 'targets' will be None
            logits, loss = self(idx)
            # focus only on the last time step
            # When calling self(idx), it should return logits in (B, T, C) shape.
            # The logits from the forward pass are already reshaped to (B*T, C).
            # We need to reshape them back to (B, T, C) to select the last time step.
            B, T_current = idx.shape
            logits = logits.reshape(B, T_current, -1)[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits,loss = m(xb,yb)
print(logits.shape)
print(loss)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [109]:
optimizer = torch.optim.AdamW(m.parameters(),lr = 1e-3)

In [110]:
batch_size = 32
for steps in range(100):
  xb,yb = get_batch('train')
  logits,loss = m(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())


4.65630578994751


In [111]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype = torch.long),max_new_tokens = 500)[0].tolist()))


oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


In [112]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a/torch.sum(a,1,keepdim = True)
b = torch.randint(0,10,(3,2)).float()
c = a@b
print('a =')
print(a)
print('--')
print(b)
print('--')
print('c=')
print(c)

a =
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [113]:
torch.manual_seed(1337)
B,T,C = 4,8,2 #batch,time,channel
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [114]:
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1]
    xbow[b,t] = torch.mean(xprev,0)


In [115]:
wei = torch.tril(torch.ones(T,T))
wei= wei/wei.sum(1,keepdim = True)
xbow2 = wei @x
torch.allclose(xbow,xbow2)

False

In [116]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril ==0,float('-inf'))
wei = F.softmax(wei,dim = -1)
xbow3 = wei @ x
torch.allclose(xbow,xbow3)

False

In [117]:
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C,head_size,bias = False)
query = nn.Linear(C,head_size,bias = False)
value = nn.Linear(C,head_size,bias = False)

k = key(x)
q = query(x)
wei = q @ k.transpose(-2,-1)



tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril ==0,float('-inf'))
wei = F.softmax(wei,dim = - 1)
v = value(x)
out = wei  @ v
out.shape


torch.Size([4, 8, 16])

In [118]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [119]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2,-1) * head_size**-0.5

In [120]:
print(wei.var(),q.var(),k.var())


tensor(1.0918) tensor(1.0700) tensor(1.0449)


In [121]:
torch.softmax(torch.tensor([0.1,-0.2,0.3,-0.2,0.5]),dim = -1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [122]:
torch.softmax(torch.tensor([0.1,-0.2,0.3,-0.2,0.5])*8,dim = -1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [123]:
class LayerNorm1d:
  def __init__(self,dim,eps = 1e-5,momentum  = 0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
  def __call__(self,x):
    xmean = x.mean(1,keepdim=True)
    xvar = x.var(1,keepdim=True)
    xhat = (x -xmean)/torch.sqrt(xvar+self.eps)
    self.out = self.gamma * xhat +self.beta
    return self.out
  def parameters(self):
    return [self.gamma,self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32,100)
x = module(x)
x.shape

torch.Size([32, 100])

In [124]:
x[:,0].mean(),x[:,0].std()


(tensor(0.1469), tensor(0.8803))

In [125]:
x[0,:].mean(),x[0,:].std()

(tensor(-9.5367e-09), tensor(1.0000))